In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 设置随机种子保证可复现性
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# 配置路径和超参数
TRAIN_CSV = "train.csv"        # 训练集路径
A_TEST_CSV = "A.csv"           # A榜测试集路径
B_TEST_CSV = "B.csv"           # B榜测试集路径
OUTPUT_A = "A_predict.csv"     # A榜预测输出
OUTPUT_B = "B_predict.csv"     # B榜预测输出

BATCH_SIZE = 1024
EPOCHS = 2
LEARNING_RATE = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 读取训练数据
train_df = pd.read_csv(TRAIN_CSV)
print(f"训练集样本数: {len(train_df)}")

# 解析 coords 和 labels 列
def parse_coords(coords_str):
    """将逗号分隔的坐标字符串转换为 (L, 3) 的 numpy 数组"""
    values = np.array([float(x) for x in coords_str.split(',')])
    return values.reshape(-1, 3)

def parse_labels(labels_str):
    """将逗号分隔的标签字符串转换为一维 numpy 数组"""
    return np.array([int(x) for x in labels_str.split(',')])

# 收集所有残基的 (坐标, 标签) 对
all_X = []   # 每个元素为 (3,) 坐标
all_y = []   # 每个元素为标签 (0-19)
for idx, row in train_df.iterrows():
    coords = parse_coords(row['coords'])
    labels = parse_labels(row['labels'])
    all_X.extend(coords)
    all_y.extend(labels)

all_X = np.array(all_X)
all_y = np.array(all_y)
print(f"总残基数: {len(all_X)}")
print(f"坐标形状: {all_X.shape}, 标签形状: {all_y.shape}")

In [ ]:
# 标准化坐标特征
scaler = StandardScaler()
all_X = scaler.fit_transform(all_X)

X_train, X_val, y_train, y_val = train_test_split(
    all_X, all_y, test_size=0.1, random_state=42, stratify=all_y
)
print(f"训练残基数: {len(X_train)}, 验证残基数: {len(X_val)}")

In [ ]:
class ResidueDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = ResidueDataset(X_train, y_train)
val_dataset = ResidueDataset(X_val, y_val)
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=4,          # 启用多进程加载
    pin_memory=True,        # 加速GPU数据传输
    persistent_workers=True # 避免每epoch重建进程
)
val_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    num_workers=4,          # 启用多进程加载
    pin_memory=True,        # 加速GPU数据传输
    persistent_workers=True # 避免每epoch重建进程
)

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=64, output_dim=20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

model = SimpleMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# 训练循环
best_val_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    avg_loss = total_loss / len(train_dataset)
    
    # 验证
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    val_acc = correct / total
    print(f"Epoch {epoch+1:2d}/{EPOCHS}, Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")

print(f"训练完成，最佳验证准确率: {best_val_acc:.4f}")

In [ ]:
# 加载最佳模型用于预测
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

In [ ]:
# 预测函数：对单个蛋白质的坐标进行预测，返回标签列表
def predict_protein(coords_str, scaler, model):
    coords = parse_coords(coords_str)          # (L, 3)
    coords_scaled = scaler.transform(coords)   # (L, 3)
    X_tensor = torch.tensor(coords_scaled, dtype=torch.float32)
    X_tensor = X_tensor.to(device)             # 将张量移动到设备
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)               # (L, 20)
        preds = torch.argmax(logits, dim=1)    # (L,)
    return preds.cpu().numpy().tolist()        # 移回 CPU 并转换为列表

In [ ]:
# 对 A 榜进行预测
print("预测 A 榜...")
a_df = pd.read_csv(A_TEST_CSV)
a_predictions = []
for idx, row in a_df.iterrows():
    pred_labels = predict_protein(row['coords'], scaler, model)
    a_predictions.append(','.join(map(str, pred_labels)))
a_out = pd.DataFrame({'domain_id': a_df['domain_id'], 'predictions': a_predictions})
a_out.to_csv(OUTPUT_A, index=False)
print(f"A 榜预测完成，保存至 {OUTPUT_A}")

In [ ]:
# 对 B 榜进行预测
print("预测 B 榜...")
b_df = pd.read_csv(B_TEST_CSV)
b_predictions = []
for idx, row in b_df.iterrows():
    pred_labels = predict_protein(row['coords'], scaler, model)
    b_predictions.append(','.join(map(str, pred_labels)))
b_out = pd.DataFrame({'domain_id': b_df['domain_id'], 'predictions': b_predictions})
b_out.to_csv(OUTPUT_B, index=False)
print(f"B 榜预测完成，保存至 {OUTPUT_B}")